# 01 - data acquisition

**purpose**: explore ravdess audio files, create speaker-disjoint train/val/test split, verify integrity, and export label files for downstream notebooks.

**input**: `data/raw/` (original zenodo download structure)

**output**: `data/{train,val,test}/{speech,song}/Actor_XX/*.wav` + `data/processed/split_labels.csv`

In [1]:
import sys
sys.path.append('../../')


**import helper modules**. we bring in the project's settings (hyperparameters, paths) and the custom data acquisition helpers. these live in `src/utils/data_acquisition.py` and handle ravdess-specific file parsing.

In [2]:
import shutil
from pathlib import Path
import pandas as pd
from src.config.config import settings
from src.utils.data_acquisition import count_wavs, collect_wavs, assign_split, _get_dest

## 1. raw data structure

we only use audio-only files (modality=03). video files are ignored.

speech wavs

In [3]:
print(f"speech wavs: {count_wavs(settings.RAW_SPEECH_DIR)}")

speech wavs: 1440


song wavs

In [4]:
print(f"song wavs: {count_wavs(settings.RAW_SONG_DIR)}")

song wavs: 1012


**total audio files**. adding speech and song counts gives us the full dataset size.

In [5]:
print(f"total speech and song wavs: {count_wavs(settings.RAW_SPEECH_DIR) + count_wavs(settings.RAW_SONG_DIR)}")

total speech and song wavs: 2452


## 2. parse filenames into metadata

filename format: `03-CHANNEL-EMOTION-INTENSITY-STATEMENT-REPETITION-ACTOR.wav`

see `src.utils.data_acquisition.parse_filename` and `src.config.config.EMOTION_MAP` for details.

In [6]:
df_speech = collect_wavs(settings.RAW_SPEECH_DIR)
df_speech.head()

,filepath,channel,emotion_code,emotion,intensity,statement,repetition,actor,gender
0,E:\career\projects\lightweight-speech-emotion-...,speech,1,neutral,normal,1,1,1,male
1,E:\career\projects\lightweight-speech-emotion-...,speech,1,neutral,normal,1,2,1,male
2,E:\career\projects\lightweight-speech-emotion-...,speech,1,neutral,normal,2,1,1,male
3,E:\career\projects\lightweight-speech-emotion-...,speech,1,neutral,normal,2,2,1,male
4,E:\career\projects\lightweight-speech-emotion-...,speech,2,calm,normal,1,1,1,male


**parse song files**. same parsing for the song channel. the filename format is identical so the same helper works.

In [7]:
df_song = collect_wavs(settings.RAW_SONG_DIR)
df_song.head()

,filepath,channel,emotion_code,emotion,intensity,statement,repetition,actor,gender
0,E:\career\projects\lightweight-speech-emotion-...,song,1,neutral,normal,1,1,1,male
1,E:\career\projects\lightweight-speech-emotion-...,song,1,neutral,normal,1,2,1,male
2,E:\career\projects\lightweight-speech-emotion-...,song,1,neutral,normal,2,1,1,male
3,E:\career\projects\lightweight-speech-emotion-...,song,1,neutral,normal,2,2,1,male
4,E:\career\projects\lightweight-speech-emotion-...,song,2,calm,normal,1,1,1,male


**merge speech and song**. we concatenate both dataframes into a single unified dataframe with all 2452 files.

In [8]:
df_all = pd.concat([df_speech, df_song], ignore_index=True)
print(f"parsed: {len(df_speech)} speech + {len(df_song)} song = {len(df_all)} total")

parsed: 1440 speech + 1012 song = 2452 total


emotion distribution:

In [9]:
print(df_all["emotion"].value_counts().sort_index())


emotion
angry      376
calm       376
fearful    376
happy      376
neutral    188
sad        376
unknown    384
Name: count, dtype: int64


**list all actors**. the dataset has 24 actors (12 male, 12 female). we confirm the full range.

In [10]:
print(f"actors: {sorted(df_all['actor'].unique())}")

actors: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24)]


**confirm 24 unique actors**. a quick sanity check that we have exactly 24 distinct speakers.

In [11]:
print(f"counts of actors: {df_all['actor'].nunique()}")

counts of actors: 24


**check gender balance**. we group by gender and count unique actors per gender, confirming a perfect 12/12 female/male split - important for bias evaluation.

In [12]:
print(f"gender split: {df_all.groupby('gender')['actor'].nunique().to_dict()}")

gender split: {'female': 12, 'male': 12}


## 3. speaker-disjoint split

by actor id (not random files) to test new speakers.

| split | actors | speech | song |
|-------|--------|--------|------|
| train | 01-19 | 19 actors | 18 actors (no actor 18 in song) |
| val   | 20-22 | 3 actors  | 3 actors |
| test  | 23-24 | 2 actors  | 2 actors |

In [13]:
df_all["split"] = df_all["actor"].apply(assign_split)
df_all.head()

,filepath,channel,emotion_code,emotion,intensity,statement,repetition,actor,gender,split
0,E:\career\projects\lightweight-speech-emotion-...,speech,1,neutral,normal,1,1,1,male,train
1,E:\career\projects\lightweight-speech-emotion-...,speech,1,neutral,normal,1,2,1,male,train
2,E:\career\projects\lightweight-speech-emotion-...,speech,1,neutral,normal,2,1,1,male,train
3,E:\career\projects\lightweight-speech-emotion-...,speech,1,neutral,normal,2,2,1,male,train
4,E:\career\projects\lightweight-speech-emotion-...,speech,2,calm,normal,1,1,1,male,train


**pivot table of split x channel**. we tabulate how many files end up in each split and channel combination. this confirms the distribution is reasonable.

In [14]:
pivot = df_all.groupby(["split", "channel"]).size().unstack(fill_value=0)
pivot["total"] = pivot.sum(axis=1)
print(pivot)

channel  song  speech  total
split                       
test       88     120    208
train     792    1140   1932
val       132     180    312


## 4. copy to split directories

destination: `data/{train,val,test}/{speech,song}/Actor_XX/`

In [15]:
if count_wavs(settings.RAW_SPEECH_DIR) > 0:
    for _, row in df_all.iterrows():
        src = Path(row["filepath"])
        dest = settings.DATA_DIR / row["split"] / row["channel"] / f"Actor_{row['actor']:02d}" / src.name
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(str(src), str(dest))
    print("files copied to split directories")
else:
    print("files already copied to split directories, skipping copy step")

files copied to split directories


**verify all files arrived**. we confirm every file exists at its destination. this catches disk-full errors or permission issues during the copy step.

In [16]:
df_all["filepath"] = df_all.apply(_get_dest, axis=1)
all_ok = all(Path(p).exists() for p in df_all["filepath"])
print(f"all {len(df_all)} files verified at destination: {all_ok}")

all 2452 files verified at destination: True


**verify no actor leakage**. we assert that no actor appears in more than one split. if an actor's files ended up in two splits, the cross-speaker evaluation would be invalid.

In [17]:
actor_splits = df_all.groupby("actor")["split"].unique()
assert all(len(v) == 1 for v in actor_splits), "actor leak detected!"
print("no actor leaks between splits.")


no actor leaks between splits.


**check gender balance across splits**. we make sure the train/val/test splits each have reasonable gender representation, though some imbalance is expected given how the actors were assigned.

In [18]:
gender_pivot = df_all.groupby(["split", "gender"]).size().unstack(fill_value=0)
print("\ngender balance:")
print(gender_pivot)


gender balance:
gender  female  male
split               
test       104   104
train      892  1040
val        208   104


## 5. export labels for downstream

save `split_labels.csv` to `data/processed/`. downstream notebooks load this csv.

In [19]:
settings.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
df_all[settings.EXPORT_COLS].to_csv(settings.LABELS_FILE, index=False)
print(f"saved {len(df_all)} labels to {settings.LABELS_FILE}")

saved 2452 labels to E:\career\projects\lightweight-speech-emotion-recognition-on-open-datasets\data\processed\split_labels.csv


**verify the export**. we read the csv back and peek at the first 3 rows to confirm it was saved correctly.

In [20]:
check = pd.read_csv(settings.LABELS_FILE)
print(check.head(3))

                                            filepath  split channel  \
0  E:\career\projects\lightweight-speech-emotion-...  train  speech   
1  E:\career\projects\lightweight-speech-emotion-...  train  speech   
2  E:\career\projects\lightweight-speech-emotion-...  train  speech   

   emotion_code  emotion intensity  actor gender  statement  repetition  
0             1  neutral    normal      1   male          1           1  
1             1  neutral    normal      1   male          1           2  
2             1  neutral    normal      1   male          2           1  


**summary statistics**. final sanity check: 2452 samples confirmed, split sizes look right, speech/song counts match expectations.

In [21]:
print(f"shape: {check.shape}")
print(f"splits: {check['split'].value_counts().to_dict()}")
print(f"channels: {check['channel'].value_counts().to_dict()}")

shape: (2452, 10)
splits: {'train': 1932, 'val': 312, 'test': 208}
channels: {'speech': 1440, 'song': 1012}


## summary

- **total files**: 2452 (1440 speech + 1012 song)
- **split**: train actors 01-19, val 20-22, test 23-24
- **integrity**: no actor leaks, gender balanced
- **labels**: `data/processed/split_labels.csv` ready for eda
- **next**: `02-eda` notebook